# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


Logistic regression provides a readable reference. Random Forest tests whether nonlinear interactions improve ranking enough to justify added complexity. Probabilities are evaluated as ranking scores because the operational question is review order, not a universal yes/no cutoff.


In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()

if ROOT.name == "notebooks" and ROOT.parent.name == "work":
    ROOT = ROOT.parent.parent

result = subprocess.run(
    [sys.executable, str(ROOT / "work" / "scripts" / "ml08_model.py")],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print("STDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"ml08_model.py failed with exit code {result.returncode}"
    )

print("Script completed. Outputs are in work/outputs.")


STDOUT:
Wrote D:\EP16\flyrank-ML-internship\work\outputs\ml08_model_queue.csv
Wrote D:\EP16\flyrank-ML-internship\work\outputs\ml08_model_metrics.json
Wrote D:\EP16\flyrank-ML-internship\work\outputs\ml08_model_report.md


STDERR:

Script completed. Outputs are in work/outputs.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


Whole pseudonymous clients remain together while 20% of clients are held out with a fixed seed. This tests generalization to unseen client groups. The target uses the same ML-07 future-review definition.


In [2]:
import json

metrics = json.loads((ROOT / 'work' / 'outputs' / 'ml08_model_metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, indent=2))


{
  "best_learned_model": "logistic_regression",
  "comparison": [
    {
      "average_precision": 0.7646334898240575,
      "brier": 0.20591541333984897,
      "correct_at_50": 47.0,
      "coverage_at_50": 0.01721763085399449,
      "log_loss": 0.6030734898242527,
      "model": "logistic_regression",
      "precision_at_10": 0.9,
      "precision_at_100": 0.92,
      "precision_at_20": 0.95,
      "precision_at_50": 0.94,
      "roc_auc": 0.7843932560606091
    },
    {
      "average_precision": 0.7699652368357957,
      "brier": 0.18761339822475293,
      "correct_at_50": 46.0,
      "coverage_at_50": 0.01721763085399449,
      "log_loss": 0.5529785841929216,
      "model": "random_forest",
      "precision_at_10": 1.0,
      "precision_at_100": 0.91,
      "precision_at_20": 1.0,
      "precision_at_50": 0.92,
      "roc_auc": 0.7892096780008008
    },
    {
      "average_precision": 0.6246565719896482,
      "brier": 0.3071625344352617,
      "correct_at_50": 33.0,
      "cove

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


The same held-out rows feed every model and the ML-07 rule. Precision@K is the primary decision metric; average precision, ROC AUC, Brier score, and log loss add ranking and probability diagnostics. Budget columns show expected correct reviews at fixed capacity.


In [3]:
import pandas as pd

queue = pd.read_csv(ROOT / 'work' / 'outputs' / 'ml08_model_queue.csv')
print(queue.head(20).to_string(index=False))

 rank          content_hash_id          client_hash_id  model_score  target  baseline_flag   suggested_action  impressions_90d  ctr_feature  expected_ctr_feature
    1 content_d2eb49b1f5f3fa34 client_c182d11e4862a37d     0.952764       1           True review_for_refresh          18422.0     0.005428              0.328039
    2 content_5b9f20eb3b7f872b client_0fa64a184f18a4a0     0.936179       1           True review_for_refresh          17617.0     0.011353              0.328039
    3 content_474c280cef704f95 client_0fa64a184f18a4a0     0.922655       0           True review_for_refresh          21084.0     0.052172              0.328039
    4 content_35f38fa4e3bdd5b2 client_c182d11e4862a37d     0.907136       1           True review_for_refresh          20001.0     0.059997              0.328039
    5 content_f4a0e5c90b283626 client_c182d11e4862a37d     0.871173       1           True review_for_refresh          11447.0     0.026208              0.318098
    6 content_9eca5b76ab3b90

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


The selected model's highest-scored false positives, ranking against the rule, and feature influence are inspected. These are review-difficulty examples, not evidence of causal impact.


In [4]:
print((ROOT / 'work' / 'outputs' / 'ml08_model_report.md').read_text(encoding='utf-8'))

# ML-08 model comparison

- Feature window: `2026-01-01` to `2026-03-31`
- Label window: `2026-04-01` to `2026-04-30`
- Split: grouped by client, seed `42`, test size `20%`
- Best learned model by Precision@50: **logistic_regression**

| Model | Precision@10 | Precision@50 | Correct@50 | Brier | Log loss |
|---|---:|---:|---:|---:|---:|
| logistic_regression | 0.9000 | 0.9400 | 47.0 | 0.2059 | 0.6031 |
| random_forest | 1.0000 | 0.9200 | 46.0 | 0.1876 | 0.5530 |
| ml07_rule_baseline | 0.6000 | 0.6600 | 33.0 | 0.3072 | 11.0713 |

The queue is a review aid. Future-window fields are used for evaluation only.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
